# Predicción de test con los modelos guardados

Aplica el modelo final a `datos/test_data.jsonlines` y genera `submission/prediccion_reproducida.csv`. No entrena LightGBM ni CatBoost: usa los modelos que guardó `1_entrenamiento.ipynb` en `modelos/`.

Pasos:
1. Rearma las variables de test con las mismas funciones de `src/` que el entrenamiento. Hace falta train: las frecuencias y la categoría descrita por sus títulos usan train + test, y los submodelos de texto, los target encodings y las categorías vecinas se ajustan con todo train (versión estándar).
2. Suma las predicciones de test de los modelos entrenados en GPU (`predicciones_gpu/`).
3. Promedia los 5 LightGBM y los 5 CatBoost, hace el promedio ponderado con TabICLv2 y aplica el umbral 0,5.
4. Verifica que las matrices rearmadas son iguales a las del entrenamiento y que la predicción coincide con la submission enviada.

Tarda unos pocos minutos en CPU.

In [1]:
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display

from features import (agregar_features_duracion, agregar_features_fechas, agregar_features_texto, categorias_por_titulos,
                      leer_fotos_atributos, leer_json_crudo, normalizar_texto, valores_campos_json, variables_atributos,
                      variables_fotos, variables_patrones, variables_vendedor_categoria)
from modelado import *

import modelado
assert Path(modelado.__file__).resolve().parent == Path("src").resolve(), "Ejecutar el notebook desde la carpeta entrega/"
CONF = cargar_configuracion()
COLUMNAS_LGBM = CONF["lightgbm"]["columnas"]
COLUMNAS_CB = CONF["catboost"]["columnas"]
IDS_CRUDOS = CONF["catboost"]["ids_crudos"]
M = CONF["m_target_encoding"]
tiempos = {}
inicio_total = time.time()

## a) Carga y procesamiento

Mismo `procesar()` y `preparar_datos()` que el entrenamiento: las categorías de las variables de texto se fijan con train.

In [2]:
inicio = time.time()
X, y, X_test, ids_test, df_train, df_test = preparar_datos(
    agregar_features=lambda df: agregar_features_duracion(agregar_features_fechas(agregar_features_texto(df))),
    devolver_procesados=True)
tiempos["a) carga y procesamiento"] = time.time() - inicio
print(f"Train: {X.shape} | test: {X_test.shape}")

Train: (70000, 93) | test: (30000, 93)


## b) Variables sin target

Vendedor y categoría (frecuencias y perfiles sobre train + test), fotos y atributos del JSON, los 22 patrones y la categoría descrita por sus títulos (`cat_svd_1..16` y las 10 categorías vecinas). Para el CatBoost, los ids crudos como texto y las categóricas como texto.

In [3]:
inicio = time.time()
variables_train, variables_test = variables_vendedor_categoria(df_train, df_test)
X_test = X_test.join(variables_test)

datos_train, datos_test = leer_fotos_atributos(ARCHIVO_TRAIN), leer_fotos_atributos(ARCHIVO_TEST)
datos_train.index, datos_test.index = X.index, X_test.index
_, atributos_test = variables_atributos(datos_train, datos_test)
X_test = X_test.join(variables_fotos(datos_test)).join(atributos_test)

PATRONES = [tuple(p.split(": ", 1)) for p in CONF["patrones"]]
crudo_test = leer_json_crudo(ARCHIVO_TEST)
X_test = X_test.join(variables_patrones(valores_campos_json(crudo_test), PATRONES).set_index(X_test.index))

CAT_SVD, CAT_VECINAS, _ = categorias_por_titulos(df_train, df_test)
X_test = X_test.join(CAT_SVD.reindex(df_test["category_id"].astype(str)).set_index(X_test.index))
tiempos["b) variables sin target"] = time.time() - inicio
print(f"Variables sin target de test: {X_test.shape[1]}")

Variables sin target de test: 156


## c) Variables con target (versión estándar)

Cada calculador se ajusta con todo train y se aplica a test: target encoding del vendedor y de la categoría (m = 3), submodelos TF-IDF + logística del título (C = 1) y de la garantía (C = 4), y % de usadas de las categorías vecinas.

In [4]:
inicio = time.time()
TODO_TRAIN = np.arange(len(y))
textos = {nombre: (df_train[col].map(normalizar_texto), df_test[col].map(normalizar_texto))
          for nombre, col in [("titulo", "title"), ("warranty", "warranty")]}
CALCULADORES = [
    calculador_submodelo(submodelo_tfidf(CONF["c_submodelo_titulo"]), *textos["titulo"], y, "proba_used_titulo"),
    calculador_submodelo(submodelo_tfidf(CONF["c_submodelo_warranty"]), *textos["warranty"], y, "proba_used_warranty"),
    calculador_target_encoding(df_train["seller_id"], df_test["seller_id"], y, "vendedor", m=M),
    calculador_target_encoding(df_train["category_id"], df_test["category_id"], y, "categoria", m=M),
    calculador_te_categorias_vecinas(df_train["category_id"], df_test["category_id"], CAT_VECINAS, y, m=M),
]
ESTANDAR_TEST = pd.concat([calcular(TODO_TRAIN, None) for calcular in CALCULADORES], axis=1).set_index(X_test.index)
tiempos["c) variables con target"] = time.time() - inicio
print(f"Variables con target: {list(ESTANDAR_TEST.columns)}")

Variables con target: ['proba_used_titulo', 'proba_used_warranty', 'te_vendedor', 'conteo_vendedor_train', 'te_categoria', 'conteo_categoria_train', 'te_categorias_vecinas']


## d) Predicciones de test de los modelos entrenados en GPU

Los tres fine-tunings de BETO (su promedio es una variable del LightGBM) y TabICLv2 (tercer miembro del promedio), ajustados con todo train en Kaggle. Ver `kaggle_gpu/`.

In [5]:
def leer_gpu_test(nombre):
    test = pd.read_csv(CARPETA_GPU / f"{nombre}_test.csv").sort_values("fila")
    assert (test["fila"].to_numpy() == np.arange(len(X_test))).all() and test["proba_used"].notna().all()
    return test["proba_used"].to_numpy()


for variable, archivo in CONF["finetunes"].items():
    X_test[variable] = leer_gpu_test(f"finetune_{archivo}")
X_test["proba_used_finetune_promedio"] = X_test[list(CONF["finetunes"])].mean(axis=1)
TABICL_TEST = leer_gpu_test("tabicl_top100")

## e) Matrices de test y verificación contra el entrenamiento

Se arman con las columnas en el orden de entrenamiento. El CatBoost recibe los ids crudos y las categóricas como texto. Se comprueba que son exactamente iguales a las que guardó `1_entrenamiento.ipynb`.

In [6]:
X_test_lgbm = X_test.join(ESTANDAR_TEST)[COLUMNAS_LGBM]

X_test_cb = X_test.copy()
for col in IDS_CRUDOS:
    X_test_cb[col] = df_test[col].astype(str).to_numpy()
CATEGORICAS_CB = [c for c in COLUMNAS_CB if c in X_test and isinstance(X_test[c].dtype, pd.CategoricalDtype)]
X_test_catboost = como_texto(X_test_cb, CATEGORICAS_CB).join(ESTANDAR_TEST)[COLUMNAS_CB]

for nombre, rearmada in [("lightgbm", X_test_lgbm), ("catboost", X_test_catboost)]:
    guardada = pd.read_pickle(CARPETA_MODELOS / f"X_test_{nombre}.pkl.gz")
    pd.testing.assert_frame_equal(rearmada, guardada, check_exact=True)
    print(f"Matriz de test del {nombre}: {rearmada.shape}, igual a la del entrenamiento")

Matriz de test del lightgbm: (30000, 164), igual a la del entrenamiento


Matriz de test del catboost: (30000, 123), igual a la del entrenamiento


## f) Predicción

- **LightGBM:** promedio de los 5 modelos de la CV estándar.
- **CatBoost:** promedio de los 5 modelos de la CV estándar.
- **Promedio ponderado** con TabICLv2 usando los pesos de `modelos/pesos.json` y umbral 0,5.

In [7]:
inicio = time.time()
proba_lgbm = np.mean([lgb.Booster(model_file=CARPETA_MODELOS / f"lightgbm_fold_{k}.txt").predict(X_test_lgbm)
                      for k in range(1, N_FOLDS + 1)], axis=0)


def cargar_catboost(k):
    modelo = CatBoostClassifier()
    modelo.load_model(str(CARPETA_MODELOS / f"catboost_fold_{k}.cbm"))
    return modelo


proba_catboost = np.mean([cargar_catboost(k).predict_proba(X_test_catboost)[:, 1] for k in range(1, N_FOLDS + 1)], axis=0)

with open(CARPETA_MODELOS / "pesos.json", encoding="utf-8") as f:
    PESOS = json.load(f)
PROBA_TEST = np.column_stack([proba_lgbm, proba_catboost, TABICL_TEST]) @ np.array(PESOS["pesos"])
display(pd.Series(PESOS["pesos"], index=PESOS["modelos"]).round(4).to_frame("peso"))
guardar_submission(ids_test, PROBA_TEST, "prediccion_reproducida", umbral=PESOS["umbral"])
tiempos["f) predicción"] = time.time() - inicio

,peso
LightGBM,0.6243
CatBoost ids crudos,0.1752
TabICL top100,0.2005


Submission guardada en submission\prediccion_reproducida.csv: 30000 filas, 47.3% used


## g) Verificación contra la submission enviada a Kaggle

In [8]:
reproducida = pd.read_csv(CARPETA_SUBMISSIONS / "prediccion_reproducida.csv")
enviada = pd.read_csv(CARPETA_SUBMISSIONS / "19_atributos_categoria_titulo.csv")
assert (reproducida["ID"] == enviada["ID"]).all()
print(f"Filas idénticas: {int((reproducida['condition'] == enviada['condition']).sum())} / {len(enviada)}")

Filas idénticas: 30000 / 30000


In [9]:
tiempos["total"] = time.time() - inicio_total
display(pd.Series(tiempos).div(60).round(1).to_frame("minutos"))

,minutos
a) carga y procesamiento,0.4
b) variables sin target,0.4
c) variables con target,0.5
f) predicción,0.3
total,1.6
